# Erster Versuch: Skript fürs Visualisieren der Fabrikumgebung

In [ ]:
# nimmt immer das aktuellste Modell
import os

# Find latest AGV model
base_path_agv = './models/agv_V'
version_agv = 1
while os.path.exists(f'{base_path_agv}{version_agv}.keras'):
    version_agv += 1
version_agv -= 1  # go back to the last existing
AGV_MODEL = f'{base_path_agv}{version_agv}.keras'
print(f"Using AGV model: {AGV_MODEL}")

# Find latest STORAGE model
base_path_storage = './models/storage_V'
version_storage = 1
while os.path.exists(f'{base_path_storage}{version_storage}.keras'):
    version_storage += 1
version_storage -= 1  # go back to the last existing
STORAGE_MODEL = f'{base_path_storage}{version_storage}.keras'
print(f"Using STORAGE model: {STORAGE_MODEL}")


#alternativ: nimmm ein bestimmtes Modell:
#AGV_MODEL = './models/agv_V1.keras'
#STORAGE_MODEL = './models/storage_V1.keras'

In [ ]:
import csv
import random
from pyglet.graphics import Batch

##### Settting parameters ####

#### AVG
V_MAX = 20
AVG_MAX_CAP = 4

#### ENV
ORDER_TIME_BONUS = 300
STATION_PROCESSING_TIME = 20
STATION_COUNT = 13
TIME_FACTOR = 400
TIME_URGENT_RED = 160
TIME_URGENT_ORANGE = 320
TIME_URGENT_YELLOW = 480


### Points
ORDER_OVERDUE_MINUS = - 0.2
ORDER_OVERDUE_MINUS_EVERY_SECONDS = 20
POINTS_ORDER_FULFILLED = 1





In [ ]:
import factory_environment as fenv
import agv
import logic
import game_mechanics_4_visualization as gm_4_vis
import RL_Agent
import visualization as vis

In [ ]:
my_env = fenv.env('ttable.csv', 'station_number.csv', 'env_control_order.csv')

In [ ]:
my_agv = agv.agv()

In [ ]:
my_logic = logic.logic()

In [ ]:
### State size
# agv: Ladezustand +  ENV: order_list 
STATION_COUNT = 13
# Ladezustand + act_stat + ENV: order list
state_size_agv = 4+13+4+13
state_size_storage = 4*10 + 1
#state_size = state_size_storage

#### Action size
# Fahre zu einer der 13 Stationen
action_size_agv = 13
### Lade auf für eine der 11 Stationen
action_size_storage = 11

In [ ]:
gm = gm_4_vis.game_mechanics(my_logic, my_env, my_agv)

In [ ]:
GAMMA = 0
EPSILON = 0.001
EPSILON_MIN = 0.000000001
EPSILON_DECAY  = 0
LEARNING_RATE = 0
my_RL_agent_AGV = RL_Agent.DQNAgent(len(gm.act_state_agv), action_size_agv, GAMMA, EPSILON, EPSILON_MIN, EPSILON_DECAY, LEARNING_RATE, load = AGV_MODEL)
my_RL_agent_storage = RL_Agent.DQNAgent(len(gm.act_state_storage), action_size_storage, GAMMA, EPSILON, EPSILON_MIN, EPSILON_DECAY, LEARNING_RATE, load = STORAGE_MODEL)

In [ ]:
my_vis = vis.visualization(gm, './station_positions.csv')

In [ ]:
START_POS = 1   # Startet an Lager B

### Sonstige Training-Variablen
MAX_TIME_ONE_GAME = 6000
GAMES_PER_EPISODE = 100
EPISODE_COUNT = 100



In [ ]:
import arcade

images_src = './graphics/'
DELTA_TIME = 0.2*2
TILE_WIDTH = 256
TILE_HEIGHT = 128

SCREEN_WIDTH = 1280
SCREEN_HEIGHT = 720
SCREEN_TITLE = 'Factory'
SCALING_SPRITE = 1
SCALING = 0.35
SCALING_AGV = 0.55
SCALING_GOOD = 0.2
FONT_SIZE = 18

AGV_vel_x = TILE_WIDTH*SCALING/2/20#(2*TILE_HEIGHT*SCALING*(3)**(1/2))/(20*(5)**(1/2)) #((TILE_HEIGHT*SCALING*(5)**(1/2))/(10*(3)**(1/2)))**(1/2)
AGV_vel_y = 1/2 * AGV_vel_x






x_start = SCREEN_WIDTH/2# - TILE_WIDTH*SCALING/2;
y_start = SCREEN_HEIGHT - 220

In [ ]:
class Factory(arcade.View):  # arcade.Window
    #def __init__(self, width, height, title):
    def __init__(self):
        #super().__init__(width, height, title)
        super().__init__()
        
        # Set up several lists
        self.all_sprites = None
        
        self.delta_time = None
        self.paused = True
        self.step = None
        self.total_time = None
        #self.width=0
        #self.heigth=0
        #self.game_count = 0
        self.map_size_x = 12
        self.map_size_y = 12
        self.agv_sprite = arcade.Sprite()
        self.agv_textures_vec = []
        self.agv_act_text_vec = []
        self.agv_act_text = 0
        self.changed_when = 0
        self.station_sprite_list = arcade.SpriteList()
        self.floor_sprite_list = arcade.SpriteList()
        self.all_sprites = arcade.SpriteList()
        self.agv_good_sprites = arcade.SpriteList()
        self.animation_running = False
        self.animation_type = None
        self.animation_ongoing = False
        #self.agv_vis_load = [0, 0 , 0, 0]
        #self.agv_vis_load_goal = [0, 0, 0, 0]
        self.agv_ori = 'x'
        self.driving_dir = 'x'
        self.final_goal = None
        self.station_processing_time = 0
        self.next_iso_coords = [6,5]
        self.final_goal_iso_pos = [6,5]
        #self.relative_good_ori_y_pos = [[x[0]*SCALING, x[1]*SCALING] for x in [[-8, 45], [-48, 25], [16, +25], [-20, 7]]]
        #self.relative_good_ori_x_pos = [self.relative_good_ori_y_pos[1], self.relative_good_ori_y_pos[2], self.relative_good_ori_y_pos[3], 
        #                                self.relative_good_ori_y_pos[0] ]
        self.relative_good_ori_y_pos = [[x[0]*SCALING, x[1]*SCALING] for x in [[100, 45], [80, 5], [85, 90], [60, 45]]]
        self.relative_good_ori_x_pos = [[x[0], TILE_WIDTH*SCALING - x[1] - 25] for x in self.relative_good_ori_y_pos]
       
        self.station_info_text = Batch()
        self.color_dec = lambda time: 'red' if time < TIME_URGENT_RED else 'orange' if time < TIME_URGENT_ORANGE else 'yellow' if time < TIME_URGENT_YELLOW else 'white'
        self.color_dict = {'red': [255, 0, 0,255], 'orange': [255, 165, 0,255], 'yellow': [255, 255, 255,255], 'green': [0, 255, 0,128], 'white': [255,255,255,255]}
        self.good_load_sprites = [arcade.Sprite(), arcade.Sprite(), arcade.Sprite(), arcade.Sprite()]
        self.storage_load_animation_done = False
        self.delta_pos = lambda: self.relative_good_ori_x_pos if self.agv_ori == 'x' else self.relative_good_ori_y_pos
        gm.reset()
        self.action = None
        self.green_arrow_sprite = arcade.Sprite()
        self.game_points = 0
        self.vis_order_list = list()
        #self.init_order_list = list()
        
        
        
        self.setup() 
        
    def setup(self):
        self.paused = True
        self.total_time = 0
        self.delta_time = DELTA_TIME
        self.station_sprite_list = arcade.SpriteList()
        self.floor_sprite_list = arcade.SpriteList()
        self.all_sprites = arcade.SpriteList()
        self.agv_good_sprites = arcade.SpriteList()
        self.changed_when = 0
        self.animation_running = False
        self.animation_type = None
        self.animation_ongoing = False
        #self.agv_vis_load = [0, 0, 0, 0]
        #self.agv_vis_load_goal = [0, 0, 0, 0]
        self.agv_ori = 'x'
        self.driving_dir = 'x'
        self.final_goal = None
        self.station_processing_time = 0
        self.next_iso_coords = [6,5]
        self.final_goal_iso_pos = [6,5]
        #self.relative_good_ori_y_pos = [[-TILE_HEIGHT/32, 0], [-TILE_WIDTH/16, +TILE_HEIGHT/16], [-TILE_HEIGHT/32, +TILE_HEIGHT/16], [0, +TILE_HEIGHT/32] ]
        #self.relative_good_ori_y_pos = [[x[0]*SCALING, x[1]*SCALING] for x in [[-8, 45], [-48, 25], [16, +25], [-20, 7]]]
        #self.relative_good_ori_x_pos = [self.relative_good_ori_y_pos[1], self.relative_good_ori_y_pos[2], self.relative_good_ori_y_pos[3], 
         #                               self.relative_good_ori_y_pos[0] ]
        self.relative_good_ori_y_pos = [[x[0]*SCALING, x[1]*SCALING] for x in [[100, 45], [80, 5], [85, 90], [60, 45]]]
        self.relative_good_ori_x_pos = [[x[0], TILE_WIDTH*SCALING - x[1] - 40] for x in self.relative_good_ori_y_pos]
        #self.good_load_sprites = [arcade.Sprite(), arcade.Sprite(), arcade.Sprite(), arcade.Sprite()]
        self.storage_load_animation_done = False
        self.delta_pos = lambda: self.relative_good_ori_x_pos if self.agv_ori == 'x' else self.relative_good_ori_y_pos
        self.action = None
        self.game_points = 0

        gm.reset()
        my_vis.reset()
        self.init_order_list = [[x[0], x[1]] for x in my_vis.gm.env.act_order_list]
        self.vis_order_list = my_vis.gm.env.act_order_list
        
 
        self.green_arrow_sprite = arcade.Sprite()

        self.station_info_text = Batch()
        # Set important init variables
        arcade.set_background_color(arcade.color.BLACK)
        
        # load map and set up lists
        #map_name='01_factory_map.tmx'  #load map
        #walls_layer_name='Stationen'  # define layer name containing impenetrable walls
        #half_walls_layer_name='half_walls'  # define layer name containing barriers for robot but not for shots
        #background_name='Fussboden'  # define layer containing background

        #my_map=arcade.tilemap.read_tmx(images_src+map_name)
        #my_map=arcade.load_tilemap(images_src+map_name, scaling=SCALING)
        self.agv_sprite = arcade.Sprite('./graphics/graphics/agv_rb_1.png', SCALING_AGV)
        for i in range(5):
            self.agv_textures_vec.append(arcade.load_texture(f'./graphics/graphics/agv_rb_{i}.png'))
        for i in range(5):
            self.agv_textures_vec.append(arcade.load_texture(f'./graphics/graphics/agv_rf_{i}.png'))
        self.agv_sprite.left, self.agv_sprite.bottom = self.get_xy_isometric(6,5)
        self.agv_sprite.iso_pos = [6, 5]
        #self.agv_sprite.change_x = -AGV_vel_x
        #self.agv_sprite.change_y = -AGV_vel_y
        

        for xpos in range(12):
            for ypos in range(12): 
                floor_sprite = arcade.Sprite("./graphics/graphics/Fussboden_01.png", SCALING)
                act_sprite = floor_sprite
                #act_sprite.center_x, act_sprite.center_y = self.get_xy_isometric(xpos, ypos)
                act_sprite.left, act_sprite.bottom = self.get_xy_isometric(xpos, ypos)
                self.floor_sprite_list.append(act_sprite)
     

        self.sprite_dict = {1: 'Lager_256.png', 2: 'robot_rb_01_256.png', 20: 'conveyer_belt_rf.png', 21: 'conveyer_belt_rb.png', 
                            3: 'Loeten_256.png', 22: 'storage_rb.png', 23: 'Lager_rf_256.png', 4: 'Bearbeitungsstation_rf_256.png',
                           5: 'Bearbeitungsstation_rb_256.png', 8: 'Lackieren_256_2.png', 9: 'Druckerstation_01_256.png',
                           10: 'Bohren_01_256.png', 12: 'Testen_256.png'}
        my_map = [[0, 0, 1, 1, 0, 1,1, 0, 0, 20, 0, 0], [0, 0, 1, 0, 1, 1, 1, 0, 0, 20, 2, 0],#
                 [0,0, 1, 1, 1, 0, 1, 0, 0, 20, 0,0], [0,0, 1, 0, 0, 1, 22, 0, 0, 3, 0,0], 
                 [0,0, 1, 1, 0, 1, 1, 0,0, 20, 0,0], [0,0, 1,1,1,23,1,0,0,4,0,0], [0,0,0,0,0,0,0,0,0,0,0,0], 
                  [0,0, 0, 8, 21, 5, 0, 5, 21, 5, 21, 21], [0,0, 0, 20, 0,0,0,0,0,0,0,0],
                 [0,0, 0, 9, 21, 10, 0, 5, 21, 12, 21, 21]]
        xpos = 0
        ypos = 0

        self.good_sprites_dict = {2: 'Karton_01.png', 3: 'metall_box_01.png', 4: 'Karton_01.png', 5: 'Karton_01.png', 6: 'Karton_01.png', 
                                 7: 'metall_box_01.png', 8: 'fass_01.png', 9: 'Karton_01.png', 10: 'metall_box_01.png', 
                                 11: 'Karton_01.png', 62: 'container_01.png', 58: 'fass_01.png'}
        #stations_alpha = 200
        for elem in my_map:
            for stat in elem:
                if stat > 0:
                    act_sprite = arcade.Sprite('./graphics/graphics/'+ self.sprite_dict[stat], SCALING)
                    #act_sprite.center_x, act_sprite.center_y = self.get_xy_isometric(xpos, ypos)
                    act_sprite.left, act_sprite.bottom = self.get_xy_isometric(xpos, ypos)
                    act_sprite.iso_pos = [xpos, ypos]
                    #act_sprite.alpha = stations_alpha
                    self.station_sprite_list.append(act_sprite)
                    #print(act_sprite.position)
                ypos += 1
            ypos = 0
            xpos += 1


        sprite_watch =  arcade.Sprite('./graphics/graphics/stoppuhr_0.png', SCALING)   
        sprite_watch.left = (SCREEN_WIDTH/2 -150)
        sprite_watch.top = SCREEN_HEIGHT-12
        sprite_watch.iso_pos = [0,0]

        sprite_cup =  arcade.Sprite('./graphics/graphics/pokal_0.png', SCALING)   
        sprite_cup.left = (SCREEN_WIDTH/2 - 25)
        sprite_cup.top = SCREEN_HEIGHT-12
        sprite_cup.iso_pos = [0,0]

        self.green_arrow_sprite = arcade.Sprite('./graphics/graphics/gruener_pfeil_0.png', SCALING)
        iso_pos = my_vis.station_iso_positions[1]
        self.green_arrow_sprite.iso_pos = [iso_pos[0] + 1, iso_pos[1]]
        screen_x, screen_y = self.get_xy_isometric(iso_pos[0], iso_pos[1])
        self.green_arrow_sprite.bottom = screen_y + 2.5*TILE_HEIGHT*SCALING
        self.green_arrow_sprite.left = screen_x + 0.25*TILE_WIDTH*SCALING       
        
        ##### fill list with all sprites ######

        #for floor in self.floor_sprite_list:
        #    self.all_sprites.append(floor)
        
        for stat in self.station_sprite_list:
            self.all_sprites.append(stat)

        self.all_sprites.append(self.agv_sprite)
        
        self.all_sprites.append(sprite_watch)
        self.all_sprites.append(sprite_cup)
        self.all_sprites.sort(key=lambda x: x.iso_pos[0])
        self.all_sprites.sort(key=lambda x: x.iso_pos[1])
        self.all_sprites.append(self.green_arrow_sprite)
        
   
        self.on_update(self.delta_time)
        self.on_draw()
        

    
    def on_key_press(self, symbol, modifiers):
        """Handle user keyboard input
        Q: Quit the game
        P: Pause/Unpause the game
        Arrows: Move Up, Left, Down, Right
        Arguments:
            symbol {int} -- Which key was pressed
            modifiers {int} -- Which modifiers were pressed
        """
        if symbol == arcade.key.Q:
        # Quit immediately
            #arcade.close_window()
            self.game_over()

        if symbol == arcade.key.P:
        # Quit immediately
            #arcade.close_window()
            self.paused = not(self.paused)

    
    def on_key_release(self, symbol: int, modifiers: int):
        pass


    
    def on_update(self, delta_time: float):
        
        """Update the positions and statuses of all game objects
        If paused, do nothing
        Arguments:
            delta_time {float} -- Time since the last update
        """

        if self.paused:
            return
        
        if my_vis.gm.test_finished() and self.animation_running == False:
            self.paused = True
            return
         # Update everything

        if self.animation_running == False:  # Gerade keine Animation, es muss eine Entscheidung getroffen werden
           
            self.action = my_vis.gm.get_decision(my_RL_agent_AGV, my_RL_agent_storage)
            if my_vis.gm.active_agent == 0:  # Lager Animation - erst wenn fertig, da Aufladen immer 20 Sekunden und es gut wäre wir wüssten Zustand beim Abfahren
                if self.action > 0:
                    #self.agv_vis_load = my_vis.gm.agv.load
                    print(f'storage decision from on_update: {self.action}')
                    my_vis.gm.do_action(self.action)
                else:
                    print(f'storage decision from on_update: {self.action}')
                    self.animation_type = 'storage'
                    self.animation_running = True
            else:  #  Das AGV trifft eine Entscheidung
                if self.action == 7 and my_vis.gm.agv.act_stat == 1:  # AGV steht am Lager und will gegenüber abliefern
                    self.animation_running = True
                    self.final_goal = 7
                    self.final_goal_iso_pos = my_vis.final_handling_iso_pos_stat[7]
                    self.animation_type = 'station_processing'
                elif my_vis.gm.agv.act_stat == self.action:  # Wenn AGV zur Station fahren soll, an der es schon steht (sollte eigentlich nicht passieren)
                    self.animation_running = False
                    res = my_vis.gm.do_action(self.action)
                    #my_vis.gm.award_points(res)
                    my_vis.gm.set_active_agent(res)
                else:
                    self.animation_running = True
                    self.animation_type = 'driving'
                    self.final_goal = self.action
                    self.final_goal_iso_pos = my_vis.final_handling_iso_pos_stat[self.action]


        else:  # Animiere Fahrt, Lagerladung oder Abladung an Station
            #my_vis.agv_act_iso_pos = [self.agv_sprite.left, self.agv_sprite.bottom]
           # print(f'agv act iso pos: {my_vis.agv_act_iso_pos}')
            if self.animation_type == 'driving':  # Animiere Fahrt
                if not self.animation_ongoing:  # Wenn gerade noch eine Animation läuft warte, bis diese zu Ende
                    self.animation_ongoing = True
                    self.next_iso_coords, self.driving_dir = my_vis.next_iso_coords(self.final_goal)
                    print(f'agv is driving to next iso pos: {self.next_iso_coords}')
                    #### Bestimme Fahrtrichtung
                    if self.driving_dir == 'y':
                        change_x = - AGV_vel_x*gm.agv.get_act_vel_factor()
                        change_y = - AGV_vel_y*gm.agv.get_act_vel_factor()
                    elif self.driving_dir == '-y':
                        change_x = + AGV_vel_x*gm.agv.get_act_vel_factor()
                        change_y = + AGV_vel_y*gm.agv.get_act_vel_factor()
                    elif self.driving_dir == 'x':
                        change_x = + AGV_vel_x*gm.agv.get_act_vel_factor()
                        change_y = - AGV_vel_y*gm.agv.get_act_vel_factor()
                    else:
                        change_x = - AGV_vel_x*gm.agv.get_act_vel_factor()
                        change_y = + AGV_vel_y*gm.agv.get_act_vel_factor()
                    ### Ende Fahrtrichtungsbestimmung
                    ### Bestimmung welche Texturausrichtung###
                    self.agv_sprite.change_x = change_x
                    self.agv_sprite.change_y = change_y
                    #for sprite in self.good_load_sprites:
                    #    sprite.change_x = change_x
                    #    sprite.change_y = change_y
                    
                    
                    if self.driving_dir in ['y', '-y']:
                        self.agv_ori = 'y'
                    else:
                        self.agv_ori = 'x'

            ### Überprüfe, ob der nächste Stop erreicht wurde
                if self.driving_dir in ['y', 'x']:  # Fährt nach unten auf Monitor
                    if self.agv_sprite.bottom <= self.get_xy_isometric(self.next_iso_coords[0], self.next_iso_coords[1])[1]:  # Ist am nächsten Schritt angekommen                        
                        my_vis.agv_act_iso_pos = self.next_iso_coords  # Setze AGV exakt auf Kachel
                        self.agv_sprite.iso_pos = my_vis.agv_act_iso_pos
                        for good in self.good_load_sprites:
                            good.iso_pos = self.agv_sprite.iso_pos                        
                        self.next_iso_coords, self.driving_dir = my_vis.next_iso_coords(self.final_goal) # Bestimme nächstes Kachelziel
                        self.animation_ongoing = False
                        if my_vis.agv_act_iso_pos == self.final_goal_iso_pos:  # Ist am finalen Ziel angekommen
                            self.animation_type = 'station_processing'
                            #self.total_time = my_vis.gm.game_time  # Ziehe Differenzen in der Zeit glatt
                            
                else:   # Fährt auf Monitor nach oben
                    if self.agv_sprite.bottom >= self.get_xy_isometric(self.next_iso_coords[0], self.next_iso_coords[1])[1]:  # Ist am nächsten Schritt angekommen
                        my_vis.agv_act_iso_pos = self.next_iso_coords  # Setze AGV exakt auf Kachel
                        self.agv_sprite.iso_pos = my_vis.agv_act_iso_pos
                        for good in self.good_load_sprites:
                            good.iso_pos = self.agv_sprite.iso_pos
                        self.next_iso_coords, self.driving_dir = my_vis.next_iso_coords(self.final_goal) # Bestimme nächstes Kachelziel
                        self.animation_ongoing = False
                        if my_vis.agv_act_iso_pos == self.final_goal_iso_pos:  # Ist am finalen Ziel angekommen
                            self.animation_type = 'station_processing'
                      
            ### Ende Überprüfung, ob nächster Stop erreicht wurde
            ###################################################
            ####### Station Animation ##########################
            #####################################################
            if self.animation_type == 'station_processing':
                self.agv_sprite.change_x = 0
                self.agv_sprite.change_y = 0   
                if self.station_processing_time >= STATION_PROCESSING_TIME:   # Station processing finished
                    old_order_amount = my_vis.gm.env.act_order_list[self.final_goal][0]
                    res = my_vis.gm.do_action(self.action)
                    my_vis.gm.set_active_agent(res)
                    self.award_points_order_fulfilled(old_order_amount)
                    
                    self.station_processing_time = 0
                    #print(f'Zeit Delta t - Station: {my_vis.gm.game_time - self.total_time}')
                    self.total_time = my_vis.gm.game_time  # Ziehe Zeitdifferenzen glatt
                    to_remove = arcade.SpriteList()
                         
                    if my_vis.gm.agv.act_stat == 8:    # An Lackiererei
                        for sprite in self.agv_good_sprites:
                            if sprite.what == 8:
                                sprite.what = 58
                    elif my_vis.gm.agv.act_stat == 12:   # An Teststation wird aufgeladen!
                        for pos in range(my_vis.gm.agv.capacity):
                            if my_vis.gm.agv.load[pos] == 62:
                                good = 62
                                act_sprite = arcade.Sprite('./graphics/graphics/'+self.good_sprites_dict[good], SCALING_GOOD)                          
                                #delta_pos = lambda: self.relative_good_ori_x_pos if self.agv_ori == 'x' else self.relative_good_ori_y_pos
                                act_sprite.bottom = self.agv_sprite.bottom + self.delta_pos()[pos][0]
                                act_sprite.left = self.agv_sprite.left + self.delta_pos()[pos][1]
                                act_sprite.pos = pos
                                act_sprite.what = good
                                act_sprite.iso_pos = my_vis.agv_act_iso_pos
                                self.agv_good_sprites.append(act_sprite)
                                self.all_sprites.append(act_sprite)
                    elif my_vis.gm.agv.act_stat in [0, 1]:   # AGV an einem Lager
                        for sprite in self.agv_good_sprites:
                            if sprite.what > 50:
                                self.all_sprites.remove(sprite)
                                to_remove.append(sprite)
                    else:
                        for sprite in self.agv_good_sprites:
                            if sprite.what == my_vis.gm.agv.act_stat:
                                self.all_sprites.remove(sprite)
                                to_remove.append(sprite)
                    
                    ### remove marked sprites
                    for sprite in to_remove:
                        self.agv_good_sprites.remove(sprite)
                    self.animation_running = False    
                else:
                    self.station_processing_time += self.delta_time
                    #my_vis.agv_act_iso_pos = self.next_iso_coords         

            ###########################################################
            ######### Lager Animation #################################
            ##############################################################
            if self.animation_type == 'storage':
                self.agv_sprite.change_x = 0
                self.agv_sprite.change_y = 0      
                #for sprite in self.good_load_sprites:
                #    sprite.change_x = 0
                #    sprite.change_y = 0
                
                if self.station_processing_time >= STATION_PROCESSING_TIME:   # Station processing finished
                    #self.animation_running = False
                    self.station_processing_time = 0
                    #print(f'Zeit Delta t - Lager: {my_vis.gm.game_time - self.total_time}')
                    self.total_time = my_vis.gm.game_time  # Ziehe Zeitdifferenzen glatt
                    #res = my_vis.gm.do_action(self.action)
                    #my_vis.gm.set_active_agent(res)
                    self.animation_running = False
                    self.storage_load_animation_done = False
                    
 
                elif (self.station_processing_time >= STATION_PROCESSING_TIME/2) and (not self.storage_load_animation_done):   # Nach Halbzeit wird Visualisierung der Güter durchgeführt
                    self.storage_load_animation_done = True
                    for sprite in self.agv_good_sprites:  # Zuerst: Entferne alle sprites von agv , dann leere Sprite list
                        self.all_sprites.remove(sprite)    

                    self.agv_good_sprites = arcade.SpriteList()
                 
                    for pos in range(my_vis.gm.agv.capacity):
                        good = my_vis.gm.agv.load[pos]
                        if good == 0:
                            continue
                        act_sprite = arcade.Sprite('./graphics/graphics/'+self.good_sprites_dict[good], SCALING_GOOD)                          
                        #delta_pos = lambda: self.relative_good_ori_x_pos if self.agv_ori == 'x' else self.relative_good_ori_y_pos
                        act_sprite.bottom = self.agv_sprite.bottom + self.delta_pos()[pos][0]
                        act_sprite.left = self.agv_sprite.left + self.delta_pos()[pos][1]
                        act_sprite.pos = pos
                        act_sprite.what = good
                        act_sprite.iso_pos = my_vis.agv_act_iso_pos
                        #act_sprite.change_x = self.agv_sprite.change_x
                        #act_sprite.change_y = self.agv_sprite.change_y
                        self.agv_good_sprites.append(act_sprite)
                        self.all_sprites.append(act_sprite)                          
                    res = my_vis.gm.do_action(self.action)
                    my_vis.gm.set_active_agent(res)
                    
                else:
                    self.station_processing_time += self.delta_time
                    #my_vis.agv_act_iso_pos = self.next_iso_coords         

        if self.agv_ori == 'x':
            self.agv_act_text_vec = [self.agv_textures_vec[x] for x in range(5,10)]
        else:
            self.agv_act_text_vec = [self.agv_textures_vec[x] for x in range(0,5)]
        if self.total_time - self.changed_when > 2 :
            self.agv_act_text += 1
            self.changed_when = self.total_time
        if self.agv_act_text > 4:
            self.agv_act_text = 0
        self.agv_sprite.texture = self.agv_act_text_vec[self.agv_act_text]
       

        # move sprites according to action
        for sprite in self.all_sprites:
            sprite.left = sprite.left+sprite.change_x*self.delta_time#int(sprite.center_x + sprite.change_x * delta_time)
            sprite.bottom = sprite.bottom+sprite.change_y*self.delta_time#int(sprite.center_y + sprite.change_y * delta_time)
        
        
        for act_sprite in self.agv_good_sprites:
            act_sprite.bottom = self.agv_sprite.bottom + self.delta_pos()[act_sprite.pos][0]
            act_sprite.left = self.agv_sprite.left + self.delta_pos()[act_sprite.pos][1]
            act_sprite.iso_pos = self.agv_sprite.iso_pos


        if (self.animation_running) and (not(self.final_goal == None)):
            iso_pos = my_vis.station_iso_positions[self.final_goal]
            self.green_arrow_sprite.iso_pos = [iso_pos[0] + 1, iso_pos[1]]
            screen_x, screen_y = self.get_xy_isometric(iso_pos[0], iso_pos[1])
            self.green_arrow_sprite.bottom = screen_y + 2.5*TILE_HEIGHT*SCALING
            self.green_arrow_sprite.left = screen_x + 0.25*TILE_WIDTH*SCALING

        
        self.total_time += self.delta_time
        self.all_sprites.sort(key=lambda x: x.iso_pos[1])
        self.all_sprites.sort(key=lambda x: x.iso_pos[0])
        pos = 0
        self.track_points()
        self.update_order_list_view()
        #self.draw_gaming_info()
        self.on_draw()                  

        


    def on_draw(self):
        """Draw all game objects """
        #arcade.start_render()
        self.clear()
        self.floor_sprite_list.draw()
        self.all_sprites.draw()
        #self.station_info_text.draw()

        i = 2
        delta = 32
        start_x = 6
        start_y = SCREEN_HEIGHT - 18
        for stat_info in self.vis_order_list[2:]:
            if stat_info[0] > 0:
                arcade.draw_text(f'{i} {stat_info[0]}, t {(stat_info[1]):.0f}', 
                            x = start_x, y = start_y + (-i+2)*delta, color = self.color_dict[self.color_dec(stat_info[1])], font_size = FONT_SIZE)
            else:
                arcade.draw_text(f'{i} {stat_info[0]}, t {stat_info[1]:.0f}', 
                            x = start_x, y = start_y + (-i+2)*delta, color = self.color_dict['green'], font_size = FONT_SIZE)            
            i += 1
        good_text = ''
        for good in gm.agv.load:
            good_text += (str(good) + ', ')
        arcade.draw_text(x = 156, y = start_y - delta, text = good_text, color = [255, 255, 255], font_size = FONT_SIZE)

        ################## Total Time ####################
        arcade.draw_text(x = SCREEN_WIDTH/2 - 100, y = SCREEN_HEIGHT - 55, text = str(int(self.total_time)), font_size = FONT_SIZE)
        ################## Punkte #########################
        arcade.draw_text(x = SCREEN_WIDTH/2 + 25, y = SCREEN_HEIGHT - 55, text = str(int(self.game_points)), font_size = FONT_SIZE)
        


    def draw_gaming_info(self):
        # Bestimme Farbe in der Stationsinfo angzeigt wird, abhängig von Dringlichkeit
        color = lambda time: 'red' if time < TIME_URGENT_RED else 'orange' if time < TIME_URGENT_ORANGE else 'yellow' if time < TIME_URGENT_YELLOW else 'white'
        color_dict = {'red': [255, 0, 0, 0], 'orange': [255, 165, 0, 0], 'yellow': [255, 255, 255, 0], 
                      'green': [0, 255, 0, 128], 'white': [255, 255, 255, 0]}
        i = 2
        delta = 12
        start_x = 6
        start_y = SCREEN_HEIGHT - 6
        text = []
        for stat_info in gm.env.act_order_list[i:]:
            text.append(arcade.Text(f'Station {i}, num: {stat_info[0]}, t: {stat_info[1]:.0f}', 
                            x = start_x, y = start_y + (i-2)*delta, color = color_dict[color(stat_info[1])], batch = self.station_info_text,
                       font_size = FONT_SIZE))
            i += 1
            
        
    def update_order_list_view(self):
        vis_order_list_to_update = [x[0] > 0 for x in my_vis.gm.env.act_order_list]
        for ind, to_change, list in zip(range(STATION_COUNT), vis_order_list_to_update, my_vis.gm.env.act_order_list):
            if to_change:
                self.vis_order_list[ind][0] = my_vis.gm.env.act_order_list[ind][0]
                self.vis_order_list[ind][1] = int(self.init_order_list[ind][1] - self.total_time)
            else:
                self.vis_order_list[ind][0] = 0
                self.vis_order_list[ind][1] = my_vis.gm.env.act_order_list[ind][1]
        
        
    def game_over(self):
        self.paused = True
        arcade.close_window()

    def get_pos_from_center_non_isometric(self, center_x, center_y):
        xpos = int((center_x - TILE_WIDTH*SCALING/2)/(TILE_WIDTH*SCALING))
        ypos = int((center_y - TILE_HEIGHT*SCALING/2)/(TILE_HEIGHT*SCALING))
        print(f'xpos: {xpos}, ypos: {ypos}')
        return xpos, ypos
    
    def get_xy_isometric(self, x, y):
        x_screen = x_start + (x - y) * TILE_WIDTH*SCALING/2;
        y_screen = y_start - (x + y) * TILE_HEIGHT*SCALING/2;
        return x_screen, y_screen

    def get_xy_isometric_from_center(self, x, y):
        xpos, ypos = self.get_pos_from_center_non_isometric(x,y)
        return self.get_xy_isometric(xpos, ypos)
        
    
    def draw_tile(self, img, x, y):
        x_screen = x_start + (x - y) * TILE_WIDTH/2;
        y_screen = y_start - (x + y) * TILE_HEIGHT/2;
        return img, x_screen, y_screen

    def track_points(self):
        order_overdue = [(x[1]<0) and (x[0] > 0) for x in my_vis.gm.env.act_order_list]
        self.game_points += sum(order_overdue)*ORDER_OVERDUE_MINUS/ORDER_OVERDUE_MINUS_EVERY_SECONDS*self.delta_time
        

    def award_points_order_fulfilled(self, old_order_amount):
        #print(old_order_amount)
        num_order_fulfilled = old_order_amount - my_vis.gm.env.act_order_list[my_vis.gm.agv.act_stat][0]
        self.game_points += num_order_fulfilled*POINTS_ORDER_FULFILLED
        


In [ ]:
window = arcade.Window(SCREEN_WIDTH, SCREEN_HEIGHT, SCREEN_TITLE, resizable=True)
start_view = Factory()
start_view.setup()
window.show_view(start_view)
start_view.setup()
arcade.run()

In [ ]:
import csv

with open("./test.csv", "w") as f:
    wr = csv.writer(f)
    wr.writerows(my_env.act_order_list)

In [ ]:
with open("./test.csv", "r", newline = '\n') as f:
    csv_reader = csv.reader(f)
    list_of_csv = list(csv_reader)

In [ ]:
[[int(x[0]), int(x[1])] for x in list_of_csv]

In [ ]:
stop it ;-)

In [ ]:
pip freeze > requirements.txt 

In [ ]:
pip install arcade